# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{getattr(dataset.metadata, 'name', 'Unknown')}: {getattr(dataset.metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All Croissant entities are referenced by their `@id`.

In [ ]:
# List all RecordSets, their IDs, and contained fields and columns
record_sets = list(dataset.record_sets())
print(f"Number of RecordSets: {len(record_sets)}\n")

record_set_ids = []
for rs in record_sets:
    print(f"RecordSet name: {getattr(rs, 'name', 'N/A')}")
    print(f"  @id          : {getattr(rs, '@id', 'N/A')}")
    print(f"  Description  : {getattr(rs, 'description', '')}")
    print(f"  FileObject(s): {[f.get('@id', '') for f in getattr(rs, 'files', [])] if hasattr(rs, 'files') else []}")
    print(f"  Fields: ")
    try:
        for field in rs.fields():
            print(f"    Field name: {getattr(field, 'name', '')}")
            print(f"      @id    : {getattr(field, '@id', '')}")
            print(f"      dataType: {getattr(field, 'data_type', '')}")
            print(f"      columns: {[col.get('@id','') for col in getattr(field,'columns',[])] if hasattr(field, 'columns') else []}")
    except Exception as e:
        print('      (No fields or failed to enumerate)')
    print()
    record_set_ids.append(getattr(rs, '@id', None))

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: load each record set by its @id (as reported above)
record_sets_to_load = [rs_id for rs_id in record_set_ids if rs_id]

dataframes = {}
for rs_id in record_sets_to_load:
    print(f"Loading records for RecordSet '@id': {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            print(f"  Loaded {len(df)} records. Columns: {df.columns.tolist()}")
            dataframes[rs_id] = df
        else:
            print("  No records returned.")
    except Exception as e:
        print(f"  ERROR: {e}")
    print()

# For the main protein abundance and related analysis, we select the primary record set id if present.
main_rs_id = ''
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Main RecordSet id is: {main_rs_id}\nColumns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())
else:
    print("No RecordSet dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes filtering, normalization, and grouping by fields.

**Choose a numeric field and a categorical field by their `@id` as referenced in the overview above. Update accordingly if your dataset uses different names.**

In [ ]:
# Example: set numeric_field_id and group_field_id by inspecting the columns above
if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    columns = df.columns.tolist()

    # Try some common column name matches (customize as seen in your RecordSet output above):
    possible_numeric_fields = [col for col in columns if any(x in col.lower() for x in ['abundance', 'coverage', 'weight', 'count', 'peptide', 'mw', 'pi']) and df[col].dtype.kind in 'fi']
    if not possible_numeric_fields:
        # Fallback: select any first numeric column
        possible_numeric_fields = [col for col in columns if df[col].dtype.kind in 'fi']
    # Attempt to match group field (such as a protein name, accession, or group)
    possible_group_fields = [col for col in columns if any(x in col.lower() for x in ['group', 'accession', 'description', 'protein', 'sample'])]

    # Choose
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else columns[0]
    group_field = possible_group_fields[0] if possible_group_fields else columns[0]

    print(f"Using numeric_field: {numeric_field}")
    print(f"Using group_field: {group_field}\n")

    # Drop NA for analysis
    sub_df = df.dropna(subset=[numeric_field])

    # Example threshold: set to mean if not sure
    threshold = sub_df[numeric_field].mean() if not sub_df.empty else 0
    filtered_df = sub_df[sub_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df[[numeric_field, group_field]].head())

    # Normalize
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 0
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df.head())
else:
    print("No valid record set loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id and main_rs_id in dataframes:
    df = dataframes[main_rs_id]
    # Plot distribution of numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    # Plot boxplot by group (if there are a manageable number)
    if group_field in df.columns and df[group_field].nunique() < 25:
        plt.figure(figsize=(12,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No dataframe available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset consists of protein abundance and related features derived from human-derived extracellular vesicles, as inspected by mass spectrometry.
- Numeric fields such as abundance, coverage, molecular weight, or counts can be analyzed and visualized for trend identification.
- Grouping by protein or sample provides insight into relative abundance or presence across groups.
- Further processing and domain-specific filtering are possible using the standardized Croissant `@id` APIs and the `mlcroissant` library.